In [ ]:
%sql
-- fact price (price, net position, negative prices)
USE CATALOG energy;
SET TIME ZONE 'UTC';

INSERT OVERWRITE d2_gold.fact_price_hourly
SELECT
  p.ts_utc,
  from_utc_timestamp(p.ts_utc, 'Europe/Brussels') AS datetime_local,
  CAST(date_format(from_utc_timestamp(p.ts_utc, 'Europe/Brussels'), 'yyyyMMdd') AS INT) AS date_key,
  hour(from_utc_timestamp(p.ts_utc, 'Europe/Brussels')) AS hour_key,
  p.zone_code,
  p.price_eur_mwh,
  n.net_position_mwh,
  p.price_eur_mwh < 0 AS is_negative_price
FROM d2_silver.silver_prices_hourly p
LEFT JOIN d2_silver.silver_net_position_hourly n
  ON p.zone_code = n.zone_code AND p.ts_utc = n.ts_utc;

In [ ]:
%sql
-- fact border: spread, convergence (|spread| < 0.01 EUR/MWh), net flows and exchanges, congestion rent, adverse flows
INSERT OVERWRITE d2_gold.fact_border_hourly
WITH hours AS (
  SELECT DISTINCT ts_utc FROM d2_silver.silver_prices_hourly
),
border_hours AS (
  SELECT
    b.border_key,
    h.ts_utc,
    pa.price_eur_mwh AS price_a,
    pb.price_eur_mwh AS price_b,
    fab.flow_mwh AS flow_ab,
    fba.flow_mwh AS flow_ba,
    sab.sched_mwh AS sched_ab,
    sba.sched_mwh AS sched_ba
  FROM d2_gold.dim_border b
  CROSS JOIN hours h
  LEFT JOIN d2_silver.silver_prices_hourly pa ON pa.zone_code = b.zone_a AND pa.ts_utc = h.ts_utc
  LEFT JOIN d2_silver.silver_prices_hourly pb ON pb.zone_code = b.zone_b AND pb.ts_utc = h.ts_utc
  LEFT JOIN d2_silver.silver_flows_hourly fab ON fab.from_zone = b.zone_a AND fab.to_zone = b.zone_b AND fab.ts_utc = h.ts_utc
  LEFT JOIN d2_silver.silver_flows_hourly fba ON fba.from_zone = b.zone_b AND fba.to_zone = b.zone_a AND fba.ts_utc = h.ts_utc
  LEFT JOIN d2_silver.silver_scheduled_hourly sab ON sab.from_zone = b.zone_a AND sab.to_zone = b.zone_b AND sab.ts_utc = h.ts_utc
  LEFT JOIN d2_silver.silver_scheduled_hourly sba ON sba.from_zone = b.zone_b AND sba.to_zone = b.zone_a AND sba.ts_utc = h.ts_utc
),
spreads AS (
  SELECT
    *,
    price_a - price_b AS spread,
    price_a IS NOT NULL AND price_b IS NOT NULL AND ABS(price_a - price_b) < 0.01 AS is_converged,
    price_a IS NOT NULL AND price_b IS NOT NULL AND ABS(price_a - price_b) >= 0.01 AS is_congested,
    CASE WHEN flow_ab IS NULL AND flow_ba IS NULL THEN NULL ELSE COALESCE(flow_ab, 0) - COALESCE(flow_ba, 0) END AS net_flow,
    CASE WHEN sched_ab IS NULL AND sched_ba IS NULL THEN NULL ELSE COALESCE(sched_ab, 0) - COALESCE(sched_ba, 0) END AS net_sched
  FROM border_hours
)
SELECT
  ts_utc,
  from_utc_timestamp(ts_utc, 'Europe/Brussels') AS datetime_local,
  CAST(date_format(from_utc_timestamp(ts_utc, 'Europe/Brussels'), 'yyyyMMdd') AS INT) AS date_key,
  hour(from_utc_timestamp(ts_utc, 'Europe/Brussels')) AS hour_key,
  border_key,
  price_a AS price_a_eur_mwh,
  price_b AS price_b_eur_mwh,
  spread AS spread_a_b_eur_mwh,
  ABS(spread) AS abs_spread_eur_mwh,
  is_converged,
  is_congested,
  flow_ab AS flow_a_to_b_mwh,
  flow_ba AS flow_b_to_a_mwh,
  net_flow AS net_flow_a_to_b_mwh,
  sched_ab AS sched_a_to_b_mwh,
  sched_ba AS sched_b_to_a_mwh,
  net_sched AS net_sched_a_to_b_mwh,
  -- estimated congestion rent = |spread| x |net day-ahead exchange| (an approximation under flow-based)
  CASE WHEN is_congested THEN ABS(spread) * ABS(net_sched) ELSE 0 END AS congestion_rent_eur,
  -- adverse flow: the net physical flow goes from the expensive zone to the cheap one
  COALESCE(is_congested AND SIGN(spread) * net_flow > 1, false) AS is_adverse_flow
FROM spreads;

In [ ]:
%sql
-- fact route flow: each border in both directions, for the flow map
INSERT OVERWRITE d2_gold.fact_route_flow_hourly
WITH routes AS (
  SELECT ts_utc, concat(b.zone_a, '>', b.zone_b) AS route_key,
         flow_a_to_b_mwh AS flow_mwh, flow_b_to_a_mwh AS reverse_flow_mwh,
         sched_a_to_b_mwh AS sched_mwh, sched_b_to_a_mwh AS reverse_sched_mwh
  FROM d2_gold.fact_border_hourly f
  JOIN d2_gold.dim_border b ON f.border_key = b.border_key
  UNION ALL
  SELECT ts_utc, concat(b.zone_b, '>', b.zone_a),
         flow_b_to_a_mwh, flow_a_to_b_mwh,
         sched_b_to_a_mwh, sched_a_to_b_mwh
  FROM d2_gold.fact_border_hourly f
  JOIN d2_gold.dim_border b ON f.border_key = b.border_key
)
SELECT
  ts_utc,
  from_utc_timestamp(ts_utc, 'Europe/Brussels') AS datetime_local,
  CAST(date_format(from_utc_timestamp(ts_utc, 'Europe/Brussels'), 'yyyyMMdd') AS INT) AS date_key,
  hour(from_utc_timestamp(ts_utc, 'Europe/Brussels')) AS hour_key,
  route_key,
  flow_mwh,
  reverse_flow_mwh,
  GREATEST(COALESCE(flow_mwh, 0) - COALESCE(reverse_flow_mwh, 0), 0) AS net_flow_mwh,
  sched_mwh,
  reverse_sched_mwh,
  GREATEST(COALESCE(sched_mwh, 0) - COALESCE(reverse_sched_mwh, 0), 0) AS net_sched_mwh
FROM routes
WHERE flow_mwh IS NOT NULL OR reverse_flow_mwh IS NOT NULL OR sched_mwh IS NOT NULL OR reverse_sched_mwh IS NOT NULL;

In [ ]:
%sql
-- fact region: CWE convergence and position of Belgium (hours where the 4 CWE zones have a price)
INSERT OVERWRITE d2_gold.fact_region_hourly
WITH cwe AS (
  SELECT p.ts_utc, p.zone_code, p.price_eur_mwh
  FROM d2_gold.fact_price_hourly p
  JOIN d2_gold.dim_zone z ON p.zone_code = z.zone_code AND z.is_cwe
),
gaps AS (
  -- gap between each price and the next lower one: a gap >= 0.01 starts a new price group
  SELECT ts_utc, zone_code, price_eur_mwh,
         price_eur_mwh - LAG(price_eur_mwh) OVER (PARTITION BY ts_utc ORDER BY price_eur_mwh) AS gap
  FROM cwe
),
stats AS (
  SELECT
    ts_utc,
    COUNT(*) AS n_zones,
    MIN(price_eur_mwh) AS price_min,
    MAX(price_eur_mwh) AS price_max,
    1 + SUM(CASE WHEN gap >= 0.01 THEN 1 ELSE 0 END) AS n_price_groups,
    MAX(CASE WHEN zone_code = 'BE' THEN price_eur_mwh END) AS be_price
  FROM gaps
  GROUP BY ts_utc
)
SELECT
  ts_utc,
  'CWE' AS region_code,
  price_min AS price_min_eur_mwh,
  price_max AS price_max_eur_mwh,
  price_max - price_min AS max_spread_eur_mwh,
  CAST(n_price_groups AS INT) AS n_price_groups,
  price_max - price_min < 0.01 AS is_full_convergence,
  be_price AS be_price_eur_mwh,
  price_max - price_min >= 0.01 AND be_price >= price_max - 0.01 AS be_is_highest,
  price_max - price_min >= 0.01 AND be_price <= price_min + 0.01 AS be_is_lowest,
  from_utc_timestamp(ts_utc, 'Europe/Brussels') AS datetime_local,
  CAST(date_format(from_utc_timestamp(ts_utc, 'Europe/Brussels'), 'yyyyMMdd') AS INT) AS date_key,
  hour(from_utc_timestamp(ts_utc, 'Europe/Brussels')) AS hour_key
FROM stats
WHERE n_zones = (SELECT COUNT(*) FROM d2_gold.dim_zone WHERE is_cwe);